# 09t Label repetitions and the map's share of the headroom (THINGS-EEG2)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 01 and 03b, and the THINGS-EEG2 dataset `cross-subject-brain-encoding-eeg` (09g).

Development analysis, not a prespecified test. The design of 09s, with the new person's labelled responses averaged over 1, 2 or all 4 repetitions; borrowing
and everything else unchanged. Does the map's gain, and the share of the headroom it recovers, grow with cleaner
labels?

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '09t-labelreps', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"ks": [250, 500], "draws": [13, 14, 15], "K": 200, "half_seed": 385, "label_reps": [1, 2, 4]}')
PEOPLE = ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10']
TAG = '09t'

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.eval import eeg1_h4b as h4b
from cross_subject_encoding.eval import h4b_analysis as ha
from cross_subject_encoding.eval import lowrank_map as lm
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.utils import config as C

import shutil
from cross_subject_encoding.data.splits import subset
with np.load(C.find_artifact("paper_alexnet_kpca1000.npz")) as f:
    z_train, z_test = f["train"].astype(np.float64), f["test"].astype(np.float64)
subsets = json.loads(C.find_artifact(C.EEG_K_SUBSETS).read_text())
people = PEOPLE
ATTACHED = te.attached(C.EEG_RAW)  # the 09g dataset (flat names) or the release layout
WORK = C.EEG_RAW if ATTACHED else Path("/kaggle/tmp/things_eeg2")
y_train, bio_test, y_reps = {}, {}, {}
for s in people:
    if not ATTACHED and te.subject_file(WORK, s, "test") is None:
        te.download_subject(s, WORK)
    tr = te.load_part(te.subject_file(WORK, s, "training"))
    y_train[s], times = tr["data"].mean(1).astype(np.float32), tr["times"]
    y_reps[s] = tr["data"].astype(np.float16)  # (images, 4 repetitions, channels, times)
    bio_test[s] = te.load_part(te.subject_file(WORK, s, "test"))["data"].astype(np.float32)
    del tr
    if not ATTACHED:
        shutil.rmtree(WORK / s, ignore_errors=True)
order = lambda s, d, k: np.asarray(subset(subsets, s, d, k))
print(len(people), 'people loaded |', f'{(time.monotonic() - T0) / 60:.1f} min')

In [ ]:
n_ch, n_t = 17, len(times)
N_TRAIN = len(next(iter(y_train.values())))
win = ha.window_index(times, 100)
t3 = lambda a, n: np.asarray(a, np.float64).reshape(n, n_ch, n_t)
post = np.repeat((times >= -1e-9)[None, :], n_ch, 0).ravel()

def sources(ztr, zte):
    tr = np.empty((len(people), len(ztr), n_ch * n_t), np.float32)
    te_ = np.empty((len(people), len(zte), n_ch * n_t), np.float32)
    q = np.concatenate([ztr, zte])
    for i, s in enumerate(people):
        pred = pl.ols_paper(ztr, y_train[s].reshape(len(ztr), -1), q)
        tr[i], te_[i] = pred[:len(ztr)], pred[len(ztr):]
    return tr, te_

def similarity(pred, meas):
    # centred over the 200 test images, then Pearson over the post-onset features
    p, m = pred[:, post] - pred[:, post].mean(0), meas[:, post] - meas[:, post].mean(0)
    p = (p - p.mean(1, keepdims=True)) / (p.std(1, keepdims=True) + 1e-30)
    m = (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-30)
    return m @ p.T / p.shape[1]  # (measured, candidates)

def calibrate(r, how):
    if how == "none":
        return r
    if how == "csls":  # cross-domain similarity local scaling (Conneau et al. 2018), k = 10, as SATTC's geometric expert
        rm = np.sort(r, 1)[:, -10:].mean(1)
        rp = np.sort(r, 0)[-10:, :].mean(0)
        return 2 * r - rm[:, None] - rp[None, :]
    if how == "dual_z":  # each candidate's similarities standardized over the test responses, then each response's
        z = (r - r.mean(0)) / (r.std(0) + 1e-30)
        return (z - z.mean(1, keepdims=True)) / (z.std(1, keepdims=True) + 1e-30)
    raise ValueError(how)

def test_time_map(pred, meas):
    # label-free, transductive (the 200 test responses as a batch, no pairing used): per 100 ms window, re-mix the
    # prediction's channels so that their covariance over images and time points matches the measured responses'
    from scipy.linalg import sqrtm
    p3, m3 = t3(pred, len(pred)), t3(meas, len(meas))
    out = p3.copy()
    for w in np.unique(win):
        cols = win == w
        xp = (p3[:, :, cols] - p3[:, :, cols].mean(0)).transpose(0, 2, 1).reshape(-1, n_ch)
        xm = (m3[:, :, cols] - m3[:, :, cols].mean(0)).transpose(0, 2, 1).reshape(-1, n_ch)
        cp = np.cov(xp, rowvar=False) + 1e-6 * np.eye(n_ch)
        cm = np.cov(xm, rowvar=False) + 1e-6 * np.eye(n_ch)
        a = np.real(np.linalg.inv(sqrtm(cp)) @ sqrtm(cm))
        mean = p3[:, :, cols].mean(0)
        out[:, :, cols] = np.einsum("nct,cd->ndt", p3[:, :, cols] - mean, a) + mean
    return out.reshape(len(pred), -1)

def encoding_r(pred, meas):
    # per-output correlation across the 200 test images with the repetition average, 100-200 ms, averaged
    sel = np.repeat(((times >= 0.1) & (times < 0.2))[None, :], n_ch, 0).ravel()
    p, m = pred[:, sel] - pred[:, sel].mean(0), meas[:, sel] - meas[:, sel].mean(0)
    return float(np.mean((p * m).sum(0) / np.sqrt((p ** 2).sum(0) * (m ** 2).sum(0) + 1e-30)))

def accuracy(r):
    rank = (r > np.diag(r)[:, None]).sum(1)
    return float((rank < 1).mean() * 100), float((rank < 5).mean() * 100)

def ci(x):
    se = x.std(ddof=1) / np.sqrt(len(x))
    t = stats.t.ppf(0.975, len(x) - 1)
    return {"mean": float(x.mean()), "lower95": float(x.mean() - t * se), "upper95": float(x.mean() + t * se), "n": int(len(x))}

def r_out(p, m):
    p, m = p - p.mean(0), m - m.mean(0)
    return (p * m).sum(0) / np.sqrt((p ** 2).sum(0) * (m ** 2).sum(0) + 1e-30)

def ceiling_r(reps):
    n = reps.shape[1]
    noise = reps.var(1, ddof=1).mean(0)
    signal = np.clip(reps.mean(1).var(0, ddof=1) - noise / n, 0, None)
    return np.sqrt(signal / (signal + noise / n + 1e-30))

if y_reps[people[0]].shape[1] < max(FIXED["label_reps"]):
    raise ValueError(f"training data have {y_reps[people[0]].shape[1]} repetitions")
N_TEST = len(z_test)
src_train, src_test = sources(z_train, z_test)
rows, per_output = [], {}
for i, person in enumerate(people):
    t = time.monotonic()
    others = [j for j in range(len(people)) if j != i]
    avg_train, avg_test = src_train[others].mean(0).astype(np.float64), src_test[others].mean(0).astype(np.float64)
    mean, basis = lm.prediction_basis(avg_train, FIXED["K"])
    reps = bio_test[person].reshape(N_TEST, bio_test[person].shape[1], -1).astype(np.float64)
    meas = reps.mean(1)
    half = np.random.default_rng([FIXED["half_seed"], i + 1]).permutation(N_TEST)
    ha, hb = half[:N_TEST // 2], half[N_TEST // 2:]
    preds = {"avg": avg_test}
    for n in FIXED["label_reps"]:
        for k in FIXED["ks"]:
            h, l = [], []
            for d in FIXED["draws"]:
                idx = order(person, d, k)
                # the new person's labels: the mean of their first n repetitions of each labelled image
                y_k = np.asarray(y_reps[person][idx, :n], np.float64).mean(1)
                h.append(h4b.apply_map(h4b.fit_map(t3(avg_train[idx], k), y_k, win, seed=d)[0], t3(avg_test, N_TEST), win).reshape(N_TEST, -1))
                l.append(lm.apply_lowrank_map(lm.fit_lowrank_map(avg_train[idx], y_k.reshape(k, -1), mean, basis, seed=d), avg_test, mean, basis))
            preds[f"h4b_k{k}_n{n}"], preds[f"lowrank_k{k}_n{n}"] = np.mean(h, 0), np.mean(l, 0)
    out = {"ceiling": ceiling_r(reps), "ceiling_a": ceiling_r(reps[ha])}
    for name, pr in preds.items():
        out[name], out[name + "_a"], out[name + "_b"] = r_out(pr, meas), r_out(pr[ha], meas[ha]), r_out(pr[hb], meas[hb])
    per_output[person] = out
    np.savez(C.METRICS_DIR / f"09t_outputs_{person}.npz", times=times, n_ch=n_ch, **out)
    rows.append({"person": person, "ceiling_post": float(out["ceiling"][post].mean()),
                 **{f"{m}_r_post": float(out[m][post].mean()) for m in preds}})
    print(person, {k: round(v, 4) if isinstance(v, float) else v for k, v in rows[-1].items()}, f"{time.monotonic() - t:.0f} s |",
          f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)
pd.DataFrame(rows).to_csv(C.METRICS_DIR / "09t_people.csv", index=False)

In [ ]:
def fit(gain, ceil, borrow):
    x = np.column_stack([np.ones_like(ceil), ceil, borrow])
    coef, *_ = np.linalg.lstsq(x, gain, rcond=None)
    r2 = 1 - ((gain - x @ coef) ** 2).sum() / ((gain - gain.mean()) ** 2).sum()
    return {"intercept": float(coef[0]), "ceiling": float(coef[1]), "borrowing": float(coef[2]), "r2": float(r2)}
maps = [f"{m}_k{k}_n{n}" for m in ("h4b", "lowrank") for k in FIXED["ks"] for n in FIXED["label_reps"]]
res = {"post_hoc": "development on THINGS-EEG2; post-onset outputs", "pooled": {}, "per_person_gain": {}, "people": rows}
for m in maps:
    pool = {k: [] for k in ("gain", "ceil", "borrow", "gain_b", "head_a")}
    sh, gains, shares = [], {}, []
    for person, o in per_output.items():
        gain, head = (o[m] - o["avg"])[post], (o["ceiling"] - o["avg"])[post]
        gain_b, head_a = (o[m + "_b"] - o["avg_b"])[post], (o["ceiling_a"] - o["avg_a"])[post]
        sh.append(float(np.corrcoef(gain_b, head_a)[0, 1]))
        gains[person] = float(gain.mean())
        shares.append(float(gain.mean() / max(head.mean(), 1e-6)))
        for k, v in (("gain", gain), ("ceil", o["ceiling"][post]), ("borrow", o["avg"][post]), ("gain_b", gain_b), ("head_a", head_a)):
            pool[k].append(v)
    P = {k: np.concatenate(v) for k, v in pool.items()}
    res["per_person_gain"][m] = gains
    res["pooled"][m] = {"fit": fit(P["gain"], P["ceil"], P["borrow"]), "split_half_corr": float(np.corrcoef(P["gain_b"], P["head_a"])[0, 1]),
                        "split_half_positive": int((np.array(sh) > 0).sum()), "mean_gain": float(P["gain"].mean()),
                        "share_of_headroom": float(np.mean(shares))}
# the test: does the gain grow with the number of label repetitions, person by person?
res["trend"] = {}
for m in ("h4b", "lowrank"):
    for k in FIXED["ks"]:
        g = np.array([[res["per_person_gain"][f"{m}_k{k}_n{n}"][p] for n in FIXED["label_reps"]] for p in per_output])
        res["trend"][f"{m}_k{k}"] = {"mean_gain_by_reps": g.mean(0).round(5).tolist(),
                                     "people_4_above_1": int((g[:, -1] > g[:, 0]).sum()),
                                     "people_monotone": int(((np.diff(g, axis=1)) > 0).all(1).sum()),
                                     "ceiling_coef_by_reps": [round(res["pooled"][f"{m}_k{k}_n{n}"]["fit"]["ceiling"], 4) for n in FIXED["label_reps"]],
                                     "share_by_reps": [round(res["pooled"][f"{m}_k{k}_n{n}"]["share_of_headroom"], 4) for n in FIXED["label_reps"]]}
tc = {"times": times.tolist(), "headroom": np.mean([(o["ceiling"] - o["avg"]).reshape(n_ch, -1).mean(0) for o in per_output.values()], 0).tolist()}
for m in maps:
    tc["gain_" + m] = np.mean([(o[m] - o["avg"]).reshape(n_ch, -1).mean(0) for o in per_output.values()], 0).tolist()
res["time_course"] = tc
(C.METRICS_DIR / "09t_summary.json").write_text(json.dumps(res, indent=2))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ms = np.array(times) * 1000
for a, m in zip(ax, ("h4b", "lowrank")):
    a.plot(ms, tc["headroom"], color="#C8102E", ls="--", label="headroom")
    for n, colour in zip(FIXED["label_reps"], ("#BDD7E7", "#6BAED6", "#08519C")):
        a.plot(ms, tc[f"gain_{m}_k{max(FIXED['ks'])}_n{n}"], color=colour, label=f"{m} k={max(FIXED['ks'])}, labels = mean of {n}")
    a.axhline(0, color="black", lw=0.5)
    a.set(xlabel="time (ms)", ylabel="r", title=f"09t: {m} gain by label repetitions")
    a.legend(fontsize=7)
fig.tight_layout()
fig.savefig(C.METRICS_DIR / "09t_figure.png", dpi=150)
print("09T_SUMMARY_JSON " + json.dumps({k: v for k, v in res.items() if k not in ("time_course", "per_person_gain")}))